# 93 — Build empirical source time functions for SPECFEM

This notebook turns catalog-selected traces into exportable source-time functions for SPECFEM tests. It creates representative hammer, PEG, and Betsy wavelets from short-body and effective-wavelet windows.

Important: these are empirical *effective* wavelets unless the short body-wave window is cleanly isolated.

In [ ]:
from pathlib import Path
import sqlite3
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, Trace, UTCDateTime
from scipy.signal import detrend
from scipy.signal.windows import tukey

# Adjust if your repo lives somewhere else
REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "lib").exists():
        REPO_ROOT = parent
        break

import sys
LIB = REPO_ROOT / "lib"
if str(LIB) not in sys.path:
    sys.path.append(str(LIB))

try:
    from segy_tools.gather import plot_wiggle_gather_from_stream, plot_wiggle_gather_from_segy
except Exception as e:
    print("Could not import segy_tools plotting helpers:", e)

In [ ]:
# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
OUTPUT_ROOT = PROJECT_ROOT / "source_waveform_analysis"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPORT_CSV = True
EXPORT_FIGURES = True

# Short window: closest approximation to early body-wave pulse.
SHORT_PRE_S = 0.005
SHORT_POST_S = 0.030

# Effective window: source + near-source ground coupling, but not full 0.5 s ground roll if avoidable.
EFFECTIVE_PRE_S = 0.010
EFFECTIVE_POST_S = 0.150

# Diagnostic window: includes ground roll/airwave/site response for inspection.
DIAGNOSTIC_PRE_S = 0.020
DIAGNOSTIC_POST_S = 0.500

WINDOWS = {
    "short_body_pulse": (SHORT_PRE_S, SHORT_POST_S),
    "effective_wavelet": (EFFECTIVE_PRE_S, EFFECTIVE_POST_S),
    "diagnostic_long": (DIAGNOSTIC_PRE_S, DIAGNOSTIC_POST_S),
}

NFFT = 8192
FMAX_PLOT = 500.0
TAPER_ALPHA = 0.5

assert CATALOG_DB.exists(), f"Catalog database not found: {CATALOG_DB}"
print(CATALOG_DB)

In [ ]:
def connect_catalog(db_path=CATALOG_DB):
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA busy_timeout = 30000")
    return conn


def read_table(conn, table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", conn)
    except Exception as e:
        print(f"Could not read {table}: {e}")
        return pd.DataFrame()


def parse_time(x):
    if pd.isna(x) or x in ("", None):
        return None
    try:
        return UTCDateTime(str(x))
    except Exception:
        try:
            return UTCDateTime(pd.to_datetime(x).to_pydatetime())
        except Exception:
            return None


def station_x_from_code(station):
    """Position-coded stations are cm along line: 08808 -> 88.08 m."""
    try:
        return float(str(station)) / 100.0
    except Exception:
        return np.nan


def load_stream_for_event(files_df, event_id, component=None, instrument_system=None):
    q = files_df[files_df["event_id"] == event_id].copy()
    if component is not None:
        q = q[q["component"].astype(str).str.upper() == component.upper()]
    if instrument_system is not None and "instrument_system" in q.columns:
        q = q[q["instrument_system"] == instrument_system]

    # Prefer 3C MiniSEED if no specific component requested.
    if component is None:
        q0 = q[(q["file_type"] == "mseed") | (q["component"].astype(str).str.upper() == "3C")]
        if len(q0):
            q = q0

    if len(q) == 0:
        raise FileNotFoundError(f"No files for event_id={event_id}, component={component}")

    # Read first available waveform file.
    row = q.iloc[0]
    f = Path(row["file_path"])
    if not f.exists():
        raise FileNotFoundError(f)
    return read(str(f))


def nearest_trace(st, source_x_m, component="Z", max_offset_m=None):
    """Return nearest Trace to source_x_m, optionally choosing a component suffix."""
    rows = []
    for i, tr in enumerate(st):
        cha = tr.stats.channel
        if component is not None and not cha.upper().endswith(component.upper()):
            continue
        x = station_x_from_code(tr.stats.station)
        if not np.isfinite(x):
            continue
        off = abs(x - source_x_m)
        if max_offset_m is not None and off > max_offset_m:
            continue
        rows.append((off, x, i, tr))
    if not rows:
        return None, None
    rows = sorted(rows, key=lambda r: r[0])
    return rows[0][3], {"offset_m": rows[0][0], "receiver_x_m": rows[0][1], "trace_index": rows[0][2]}


def get_pick_time_for_trace(picks_df, event_id, tr, preferred_phase="P"):
    if picks_df.empty:
        return None
    q = picks_df[picks_df["event_id"] == event_id].copy()
    for col, value in [("station", tr.stats.station), ("channel", tr.stats.channel)]:
        if col in q.columns:
            q = q[q[col].astype(str) == str(value)]
    if "phase" in q.columns and preferred_phase is not None:
        qp = q[q["phase"].astype(str).str.upper() == preferred_phase.upper()]
        if len(qp):
            q = qp
    for col in ["pick_time_utc", "pick_time", "consensus_time", "time"]:
        if col in q.columns and q[col].notna().any():
            return parse_time(q.iloc[0][col])
    return None


def extract_window_around_time(tr, center_time, pre_s, post_s, taper_alpha=TAPER_ALPHA):
    if center_time is None:
        raise ValueError("center_time is None")
    t1 = center_time - pre_s
    t2 = center_time + post_s
    pulse = tr.copy().trim(t1, t2, pad=True, fill_value=0)
    x = pulse.data.astype(float)
    if len(x) < 4:
        raise ValueError("window too short")
    x = detrend(x, type="linear")
    x = x - np.nanmean(x)
    w = tukey(len(x), alpha=taper_alpha)
    xt = x * w
    return pulse, x, xt


def amplitude_spectrum(x, dt, nfft=NFFT, normalize=True):
    freq = np.fft.rfftfreq(nfft, dt)
    amp = np.abs(np.fft.rfft(x, n=nfft))
    if normalize and np.nanmax(amp) > 0:
        amp = amp / np.nanmax(amp)
    return freq, amp


def plot_waveform_spectrum(x, xt, dt, title="", fmax=FMAX_PLOT):
    t = np.arange(len(x)) * dt
    freq, amp = amplitude_spectrum(xt, dt)
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
    axes[0].plot(t, x, label="raw window")
    axes[0].plot(t, xt, label="tapered")
    axes[0].set_xlabel("Time since window start (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(title)
    axes[1].plot(freq, amp)
    axes[1].set_xlim(0, fmax)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Normalized amplitude")
    axes[1].grid(True, alpha=0.3)
    return fig, freq, amp


def event_label(row):
    parts = [str(row.get("line", "")), str(row.get("source_type", "")), str(row.get("source_x_m", "")), str(row.get("event_id", ""))]
    return " | ".join([p for p in parts if p and p != "nan"])

In [ ]:
conn = connect_catalog()
shot_events = read_table(conn, "shot_events")
shot_files = read_table(conn, "shot_gather_files")
picks = read_table(conn, "picks")

shot_events["source_x_m"] = pd.to_numeric(shot_events.get("source_x_m", np.nan), errors="coerce")
print(shot_events.shape, shot_files.shape, picks.shape)

## STF extraction and stacking helpers

In [ ]:
def resample_array_to_common_dt(x, dt, target_dt):
    if abs(dt - target_dt) < 1e-9:
        return x
    from scipy.signal import resample
    duration = len(x) * dt
    n = int(round(duration / target_dt))
    return resample(x, n)


def extract_empirical_wavelets(events_df, source_type=None, instrument_system=None, window_name="effective_wavelet", max_events=100):
    pre_s, post_s = WINDOWS[window_name]
    q = events_df[events_df["source_x_m"].notna()].copy()
    if source_type is not None and "source_type" in q.columns:
        q = q[q["source_type"].astype(str).str.lower().str.contains(source_type.lower(), na=False)]
    if instrument_system is not None and "instrument_system" in q.columns:
        q = q[q["instrument_system"].astype(str).str.lower().str.contains(instrument_system.lower(), na=False)]
    q = q.head(max_events)

    wavelets = []
    meta = []
    for _, ev in q.iterrows():
        event_id = str(ev["event_id"])
        source_x_m = float(ev["source_x_m"])
        try:
            st = load_stream_for_event(shot_files, event_id, component=None)
            tr, ninfo = nearest_trace(st, source_x_m, component="Z")
            if tr is None:
                continue
            center = get_pick_time_for_trace(picks, event_id, tr)
            if center is None:
                center = tr.stats.starttime
            pulse, x, xt = extract_window_around_time(tr, center, pre_s, post_s)
            # Normalize by max abs for shape stacking, not amplitude calibration.
            denom = np.nanmax(np.abs(xt))
            if denom > 0:
                xt = xt / denom
            wavelets.append((xt, pulse.stats.delta))
            meta.append({
                "event_id": event_id,
                "instrument_system": ev.get("instrument_system", None),
                "source_type": ev.get("source_type", None),
                "source_x_m": source_x_m,
                "station": tr.stats.station,
                "channel": tr.stats.channel,
                "receiver_x_m": ninfo["receiver_x_m"],
                "offset_m": ninfo["offset_m"],
                "dt": pulse.stats.delta,
            })
        except Exception as e:
            continue
    return wavelets, pd.DataFrame(meta)


def stack_wavelets(wavelets):
    if not wavelets:
        return None, None
    target_dt = min(dt for _, dt in wavelets)
    arrs = [resample_array_to_common_dt(x, dt, target_dt) for x, dt in wavelets]
    n = min(len(a) for a in arrs)
    A = np.vstack([a[:n] for a in arrs])
    stack = np.nanmedian(A, axis=0)
    stack = stack - np.nanmean(stack)
    denom = np.nanmax(np.abs(stack))
    if denom > 0:
        stack = stack / denom
    return stack, target_dt


def export_stf(name, stack, dt, outdir):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    t = np.arange(len(stack)) * dt
    df = pd.DataFrame({"time_s": t, "amplitude": stack})
    csv_path = outdir / f"{name}.csv"
    df.to_csv(csv_path, index=False)

    tr = Trace(data=stack.astype(np.float32))
    tr.stats.delta = dt
    tr.stats.station = name[:5].upper().replace("_", "")
    tr.stats.channel = "STF"
    mseed_path = outdir / f"{name}.mseed"
    Stream([tr]).write(str(mseed_path), format="MSEED")

    freq, amp = amplitude_spectrum(stack, dt)
    spec = pd.DataFrame({"frequency_hz": freq, "normalized_amplitude": amp})
    spec_path = outdir / f"{name}_spectrum.csv"
    spec.to_csv(spec_path, index=False)
    return {"csv": csv_path, "mseed": mseed_path, "spectrum_csv": spec_path}

## Build representative STFs

In [ ]:
outdir = OUTPUT_ROOT / "empirical_stfs_for_specfem"
outdir.mkdir(parents=True, exist_ok=True)

source_types = ["hammer", "peg", "betsy"]
window_names = ["short_body_pulse", "effective_wavelet"]
summary_rows = []

for source_type in source_types:
    for window_name in window_names:
        wavelets, meta = extract_empirical_wavelets(
            shot_events,
            source_type=source_type,
            instrument_system=None,
            window_name=window_name,
            max_events=200,
        )
        stack, dt = stack_wavelets(wavelets)
        if stack is None:
            print("No wavelets for", source_type, window_name)
            continue
        name = f"{source_type}_{window_name}_median_empirical_stf"
        paths = export_stf(name, stack, dt, outdir)
        freq, amp = amplitude_spectrum(stack, dt)
        peak_freq = float(freq[np.nanargmax(amp)])
        summary_rows.append({
            "source_type": source_type,
            "window_name": window_name,
            "n_wavelets": len(wavelets),
            "dt": dt,
            "duration_s": len(stack)*dt,
            "peak_freq_hz_normalized_spectrum": peak_freq,
            **{k: str(v) for k, v in paths.items()},
        })
        meta.to_csv(outdir / f"{name}_input_events.csv", index=False)

        fig, _, _ = plot_waveform_spectrum(stack, stack, dt, title=name)
        fig.savefig(outdir / f"{name}.png", dpi=150)
        plt.close(fig)

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(outdir / "empirical_stf_exports_summary.csv", index=False)